## 离线构造筛选

新算零假设、选择与相关/协整反例，不交易。French筛选和历史扰动比较均跳过，无市场原/解压/处理文件随包，权利与执行闸门保持。

[本语言逐本运行模式与准备条件](../../../../NOTEBOOK_GUIDE.md)。运行步骤、数据要求和预期输出见本指南。

# 项目5准备层：训练窗口配对筛选（不交易）

共用screening_layer.py。默认加载规则，跳过历史45对行业筛选，再运行30次零假设和30次选择效应、相关≠协整及hedge ratio≠资本权重。CLI的200次另有身份。新写一幅零假设拒绝数图；旧“不写文件”说明不准确。无入退出、双腿成本、融资或策略PnL。

In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("p05_screening_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import importlib.util
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
spec = importlib.util.spec_from_file_location("p05", ROOT / "projects/project05/preparation/screening_layer.py")
p05 = importlib.util.module_from_spec(spec); spec.loader.exec_module(p05)


## 1. 冻结规则，历史数据跳过

In [ ]:
rules = p05.load_rules()
print("French real_levels and historical industry screening SKIPPED: no redistribution/use rights established")


## 2. 历史训练筛选跳过

后两格仅输出skip，不产生新的测试数、BH或Bonferroni结果。

In [ ]:
print("SKIPPED historical French screening/perturbation comparison; constructed examples follow")


In [ ]:
print("SKIPPED historical French screening/perturbation comparison; constructed examples follow")


## 3. 历史未来扰动比较跳过

skip不是本次完成的不可侵入性检查；训练统计量不得读未来仍是契约。

In [ ]:
print("SKIPPED historical French screening/perturbation comparison; constructed examples follow")


## 4. 零假设下的假阳性与“选最好一对”的选择效应（构造，少量重复）

In [ ]:
small = {**rules, "null_calibration": {**rules["null_calibration"], "reps": 30}, "selection_effect": {**rules["selection_effect"], "reps": 30}}
null = p05.null_calibration(small)
sel = p05.selection_effect(small)
print({k: round(v, 3) for k, v in null.items() if isinstance(v, float)})
print({k: round(v, 3) for k, v in sel.items() if isinstance(v, float)})
assert null["mean_rejections_at_alpha"] > 1.0 and sel["mean_test_pvalue_of_selected_pair"] > 0.2

## 5. 相关≠协整，hedge ratio≠资本权重

In [ ]:
cc = p05.correlation_is_not_cointegration()
print(pd.DataFrame(cc).T.round(4))
w = p05.hedge_ratio_is_not_a_capital_weight()
print({k: round(v, 4) for k, v in w.items()})
assert cc["common_trend_not_cointegrated"]["engle_granger_pvalue"] > 0.05 > cc["cointegrated"]["engle_granger_pvalue"]

In [ ]:
fig, ax = plt.subplots(figsize=(7,3))
ax.bar(["null mean rejections", "BH mean rejections"], [null["mean_rejections_at_alpha"],null["mean_bh_rejections"]])
ax.set(title="CONSTRUCTED independent random walks: multiple tests",ylabel="Mean rejected pairs")
fig.tight_layout(); fig.savefig(RUN_DIR / "screening_null_CONSTRUCTED.png",dpi=130); plt.show()


## 6. 自己试一试

1. 把 `rules["alpha"]` 想成 0.10，数一数两个窗口里“显著”的配对个数和零假设下的期望个数，说明为什么“显著个数多了”不是发现。
2. 在第 2 节再加一个训练窗口（比如训练到 1985-12），写下总检验次数与相应的 Bonferroni 阈值；这个窗口应该事前还是事后加？
3. 在第 5 节改 `hedge_ratio_is_not_a_capital_weight(beta=0.8, price_y=50, price_x=100)`，算资本权重与净美元暴露。

前两题的历史French筛选只作设计与多重检验讨论，不启用市场加载器。French 数据权利和执行授权尚未确立；第三题是独立构造资本权重手算，不替代市场实验。